# Prática 03/11 · Da fonte à base analítica: o ETL do PPI-molde
**Aprendizagem Computacional · 4º TIA · Aula 02 · versão INICIAL**

Em 27/10, definimos o problema: prever, **antes** da eleição, a abstenção de cada município de MT. Hoje construímos a **base analítica** a partir das fontes oficiais, em código, que é o que o grupo entrega no check-in de 10/11 (base v0, dicionário e EDA).

**ETL** quer dizer *Extrair, Transformar e Carregar*:
- **Extrair:** ler os arquivos do TSE e as coordenadas dos municípios;
- **Transformar:** filtrar, somar por município, calcular percentuais e juntar as fontes;
- **Carregar:** validar e salvar `base_analitica.csv`, uma linha por município.

Ao final, a base que vocês geram tem que ser **idêntica** à do PPI-molde. Isso é reprodutibilidade: qualquer pessoa, com o código, chega aos mesmos números.

**Precisa de internet** (os dados vêm do TSE e do GitHub). Use o Colab.

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | Percentuais do perfil do eleitorado por município | cap. 3 e 4 |
| 2 | Filtrar presidente, 1º turno, MT | cap. 3 |
| 3 | Somar as zonas e calcular a abstenção | cap. 3 |
| 4 | Juntar as fontes pela chave, 1 para 1 | cap. 6 |
| 5 | Distância até Cuiabá | cap. 7 |
| 6 | Tratar o município sem histórico e validar | cap. 6 |

## 1 · As fontes (pronto)
| Fonte | Arquivo | O que tiramos dele |
|---|---|---|
| TSE · perfil do eleitorado 2026 (julho) | `perfil_eleitorado_2026.zip` → `perfil_eleitorado_2026_MT.csv` | faixa etária e escolaridade dos eleitores |
| TSE · resultados 2026, por município e zona | `detalhe_votacao_munzona_2026.zip` | aptos e abstenções no 1º turno (o **y**) |
| TSE · resultados 2022, por município e zona | `detalhe_votacao_munzona_2022.zip` | abstenção da eleição anterior (um **X**) |
| IBGE · coordenadas das sedes (repositório `kelvins/municipios-brasileiros`) | `municipios.csv` | latitude e longitude |

Os ZIP do TSE são grandes (o do perfil tem cerca de 400 MB, com o Brasil inteiro). A função abaixo lê o ZIP **direto do servidor, aos pedaços**: o `zipfile` lê o índice (no fim do arquivo) e depois só o CSV que pedimos. Assim baixamos poucos MB. Não precisa entender cada linha dela; ela é uma ferramenta, como o `read_csv`.

In [ ]:
import io
import unicodedata
import urllib.request
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.precision', 2)
CABECALHO = {'User-Agent': 'Mozilla/5.0 (ETL didatico PPI GovTech IFMT)'}
URL_TSE = 'https://cdn.tse.jus.br/estatistica/sead/odsele'
URL_COORDENADAS = 'https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv'
# Arquivos gerados ficam FORA da pasta da prática: /content no Colab, sua pasta de usuário no laboratório
SAIDA = (Path('/content') if Path('/content').exists() else Path.home() / 'ac_saida') / 'aula02'
SAIDA.mkdir(parents=True, exist_ok=True)

class ArquivoRemoto(io.RawIOBase):
    """Arquivo na internet que se comporta como arquivo local: cada leitura baixa só o trecho pedido.

    Assim o `zipfile` lê o índice do ZIP (que fica no fim) e só o membro que nos interessa.
    """
    def __init__(self, url, bloco=1 << 20):
        self.url, self.pos, self.bloco, self.cache = url, 0, bloco, {}
        with urllib.request.urlopen(urllib.request.Request(url, method='HEAD', headers=CABECALHO)) as r:
            self.tamanho = int(r.headers['Content-Length'])

    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos

    def seek(self, deslocamento, origem=io.SEEK_SET):
        base = {io.SEEK_SET: 0, io.SEEK_CUR: self.pos, io.SEEK_END: self.tamanho}[origem]
        self.pos = max(0, base + deslocamento)
        return self.pos

    def _bloco(self, i):
        if i not in self.cache:
            ini, fim = i * self.bloco, min((i + 1) * self.bloco, self.tamanho) - 1
            req = urllib.request.Request(self.url, headers={**CABECALHO, 'Range': f'bytes={ini}-{fim}'})
            with urllib.request.urlopen(req) as r:
                if r.status != 206:
                    raise RuntimeError('o servidor não aceitou leitura por partes; baixe o ZIP e passe o caminho local')
                self.cache = {i: r.read()}          # guarda só o último bloco (memória baixa)
        return self.cache[i]

    def readinto(self, destino):
        n = min(len(destino), self.tamanho - self.pos)
        lidos = 0
        while lidos < n:
            i, off = divmod(self.pos, self.bloco)
            pedaco = self._bloco(i)[off:off + n - lidos]
            destino[lidos:lidos + len(pedaco)] = pedaco
            lidos += len(pedaco); self.pos += len(pedaco)
        return lidos


def ler_csv_do_zip(origem, membro, colunas):
    """Lê só as colunas necessárias de um CSV do TSE (sep ';', latin1) dentro de um ZIP.

    `origem` pode ser um caminho local ou uma URL (lida aos pedaços, sem baixar o ZIP inteiro).
    """
    arquivo = ArquivoRemoto(origem) if str(origem).startswith('http') else origem
    with zipfile.ZipFile(arquivo) as z, z.open(membro) as f:
        return pd.read_csv(io.TextIOWrapper(f, encoding='latin1'), sep=';', usecols=colunas,
                           dtype={'CD_MUNICIPIO': str})

## 2 · Extrair (pronto)
Lemos só as colunas necessárias. Leva cerca de um minuto.

In [ ]:
perfil = ler_csv_do_zip(f'{URL_TSE}/perfil_eleitorado/perfil_eleitorado_2026.zip', 'perfil_eleitorado_2026_MT.csv',
                        ['CD_MUNICIPIO', 'NM_MUNICIPIO', 'CD_FAIXA_ETARIA', 'CD_GRAU_ESCOLARIDADE', 'QT_ELEITORES'])
COLS_RES = ['SG_UF', 'CD_MUNICIPIO', 'NR_TURNO', 'CD_CARGO', 'QT_APTOS', 'QT_ABSTENCOES']
res_2026 = ler_csv_do_zip(f'{URL_TSE}/detalhe_votacao_munzona/detalhe_votacao_munzona_2026.zip',
                          'detalhe_votacao_munzona_2026_BRASIL.csv', COLS_RES)
res_2022 = ler_csv_do_zip(f'{URL_TSE}/detalhe_votacao_munzona/detalhe_votacao_munzona_2022.zip',
                          'detalhe_votacao_munzona_2022_BRASIL.csv', COLS_RES)
coordenadas = pd.read_csv(URL_COORDENADAS)
print('perfil:', perfil.shape, '| resultados 2026:', res_2026.shape, '| 2022:', res_2022.shape, '| coordenadas:', coordenadas.shape)
perfil.head(3)

Cada linha do perfil é um **grupo** de eleitores de um município e zona com a mesma faixa etária, escolaridade, gênero etc., e `QT_ELEITORES` diz quantos são. A unidade de análise que queremos é o **município**: temos que somar.

Os códigos que usamos (do `leiame.pdf` do TSE):
- faixas de 16 a 24 anos: 1600, 1700, 1800, 1900, 2000 e 2124;
- 65 anos ou mais: 6569, 7074, 7579, 8084, 8589, 9094, 9599 e 9999;
- superior completo: escolaridade 8.

## 3 · TODO 1: o perfil por município (cap. 3 e 4)
A célula já cria colunas com os eleitores de cada grupo de interesse (zero quando a linha não é do grupo) e soma tudo por município. Complete os três percentuais: 100 × grupo ÷ total (`QT_ELEITORES`). O log do tamanho já está pronto.

In [ ]:
FAIXAS_16_24 = [1600, 1700, 1800, 1900, 2000, 2124]
FAIXAS_65_MAIS = [6569, 7074, 7579, 8084, 8589, 9094, 9599, 9999]

p = perfil.assign(
    N_16_24=perfil['QT_ELEITORES'].where(perfil['CD_FAIXA_ETARIA'].isin(FAIXAS_16_24), 0),
    N_65_MAIS=perfil['QT_ELEITORES'].where(perfil['CD_FAIXA_ETARIA'].isin(FAIXAS_65_MAIS), 0),
    N_SUPERIOR=perfil['QT_ELEITORES'].where(perfil['CD_GRAU_ESCOLARIDADE'] == 8, 0))
m = p.groupby(['CD_MUNICIPIO', 'NM_MUNICIPIO'], as_index=False)[['QT_ELEITORES', 'N_16_24', 'N_65_MAIS', 'N_SUPERIOR']].sum()

base = pd.DataFrame({'CD_MUNICIPIO': m['CD_MUNICIPIO'], 'NM_MUNICIPIO': m['NM_MUNICIPIO'],
                     'ELEITORADO_PERFIL': m['QT_ELEITORES']})
base['PCT_16_24'] = None               # TODO 1: 100 × N_16_24 ÷ QT_ELEITORES (use as colunas de m)
base['PCT_65_MAIS'] = None             # TODO 1
base['PCT_SUPERIOR_COMPLETO'] = None   # TODO 1
base['LOG_ELEITORADO'] = np.log1p(m['QT_ELEITORES'])
print(len(base), 'municípios no perfil')
base.head()

## 4 · TODO 2 e TODO 3: a abstenção por município (cap. 3)
O arquivo de resultados tem o **Brasil inteiro**, todos os cargos, os dois turnos, e uma linha por **zona** eleitoral (municípios grandes têm várias zonas).

Complete a função:
- **TODO 2:** fique só com as linhas em que `SG_UF == uf`, `CD_CARGO == 1` (presidente) e `NR_TURNO == 1`. Combine as condições com `&`, cada uma entre parênteses.
- **TODO 3:** some `QT_APTOS` e `QT_ABSTENCOES` por `CD_MUNICIPIO` e calcule `ABSTENCAO_PCT = 100 × abstenções ÷ aptos`.

A mesma função serve para 2026 (o y) e para 2022 (um X).

In [ ]:
def abstencao_por_municipio(resultado, uf='MT'):
    r = None   # TODO 2: linhas da UF, cargo 1 (presidente) e turno 1
    m = None   # TODO 3: soma de QT_APTOS e QT_ABSTENCOES por CD_MUNICIPIO (groupby com as_index=False)
    m['ABSTENCAO_PCT'] = None   # TODO 3: 100 × abstenções ÷ aptos
    return m

atual = abstencao_por_municipio(res_2026)
anterior = abstencao_por_municipio(res_2022).rename(columns={'QT_APTOS': 'APTOS_2022', 'QT_ABSTENCOES': 'ABST_2022',
                                                             'ABSTENCAO_PCT': 'ABST_2022_PCT'})
print('2026:', len(atual), 'municípios | 2022:', len(anterior), 'municípios')

**Pergunta para discussão:** 2026 tem um município a mais que 2022. Qual pode ser o motivo, e o que fazer com ele?

**Resposta esperada:** um município instalado depois de 2022 (aqui, Boa Esperança do Norte) não tem resultado anterior. Ou ele sai da base, ou a abstenção de 2022 é estimada (por exemplo, pela do município de origem). Qualquer escolha serve, desde que esteja **escrita** no artigo. No molde, ele sai.

## 5 · TODO 4: juntar as fontes, 1 para 1 (cap. 6)
A chave é `CD_MUNICIPIO`, o código do TSE (texto). Nunca junte pelo **nome**: acentos e grafias mudam entre fontes.

Complete as duas junções com `base.merge(...)`:
- com `atual` (y), `how='inner'`;
- com `anterior[['CD_MUNICIPIO', 'APTOS_2022', 'ABST_2022_PCT']]`, `how='left'` (quem não tem 2022 fica com vazio, para decidirmos depois).

Use `validate='1:1'` nas duas: se a chave se repetir de um dos lados, o pandas para com erro em vez de duplicar linhas em silêncio.

In [ ]:
base = None   # TODO 4: base.merge com atual, on='CD_MUNICIPIO', how='inner', validate='1:1'
base = None   # TODO 4: base.merge com as 3 colunas de anterior, how='left', validate='1:1'
base['CRESC_ELEITORADO_PCT'] = 100 * (base['ELEITORADO_PERFIL'] / base['APTOS_2022'] - 1)
print(base.shape)
base[['NM_MUNICIPIO', 'ABSTENCAO_PCT', 'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT']].head()

## 6 · TODO 5: a distância até Cuiabá (cap. 7)
O arquivo de coordenadas não tem o código do TSE, só o do IBGE. Aqui, excepcionalmente, juntamos pelo **nome normalizado** (sem acento, em maiúsculas, sem apóstrofo) e conferimos que nenhum município ficou sem coordenada.

A distância em linha reta entre dois pontos da Terra sai da fórmula de **haversine**, já pronta em `distancia_km`. Complete: `base['DIST_CUIABA_KM'] = distancia_km(base['LATITUDE'], base['LONGITUDE'])`.

É uma distância aproximada: o eleitor vai pela estrada, não em linha reta. Isso é uma limitação para declarar.

In [ ]:
def normalizar_nome(nome):
    """'Mirassol d'Oeste' e 'MIRASSOL D OESTE' viram a mesma chave."""
    sem_acento = unicodedata.normalize('NFKD', nome).encode('ascii', 'ignore').decode()
    return ' '.join(sem_acento.upper().replace("'", ' ').replace('-', ' ').split())


def distancia_km(lat, lon, ref=(-15.601, -56.097)):
    """Distância em linha reta (haversine) até a sede de Cuiabá."""
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, ref[0], ref[1]))
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


coord = coordenadas[coordenadas['codigo_uf'] == 51].assign(CHAVE=lambda d: d['nome'].map(normalizar_nome))
base['CHAVE'] = base['NM_MUNICIPIO'].map(normalizar_nome)
base = base.merge(coord[['CHAVE', 'latitude', 'longitude']], on='CHAVE', how='left', validate='1:1')
base = base.rename(columns={'latitude': 'LATITUDE', 'longitude': 'LONGITUDE'})
assert base['LATITUDE'].notna().all(), 'há município sem coordenada'
base['DIST_CUIABA_KM'] = None   # TODO 5: distancia_km(...) com a latitude e a longitude da base
base.nlargest(3, 'DIST_CUIABA_KM')[['NM_MUNICIPIO', 'DIST_CUIABA_KM']]

## 7 · TODO 6: o município sem histórico, as checagens e o arquivo final (cap. 6)
1. Retire da base as linhas com `ABST_2022_PCT` vazio (`dropna(subset=[...])`) e refaça o índice (`reset_index(drop=True)`).
2. Rode as checagens: cada `assert` é uma regra do negócio escrita em código.
3. A célula salva a base com as colunas na ordem do dicionário.

In [ ]:
sem_historico = base.loc[base['ABST_2022_PCT'].isna(), 'NM_MUNICIPIO'].tolist()
print('sem resultado em 2022, retirados:', sem_historico)
base = None   # TODO 6: base sem as linhas com ABST_2022_PCT vazio, com o índice refeito

COLUNAS = ['CD_MUNICIPIO', 'NM_MUNICIPIO', 'ELEITORADO_PERFIL', 'PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO',
           'LOG_ELEITORADO', 'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM', 'LATITUDE', 'LONGITUDE',
           'QT_APTOS', 'QT_ABSTENCOES', 'ABSTENCAO_PCT']
base = base[COLUNAS].sort_values('CD_MUNICIPIO').reset_index(drop=True)
assert base['CD_MUNICIPIO'].is_unique, 'município repetido'
assert base.isna().sum().sum() == 0, 'há valores ausentes'
assert base['ABSTENCAO_PCT'].between(0, 100).all() and base['ABST_2022_PCT'].between(0, 100).all()
assert (base['QT_ABSTENCOES'] <= base['QT_APTOS']).all()
base.to_csv(SAIDA / 'base_analitica.csv', index=False)
print(len(base), 'municípios salvos em', SAIDA / 'base_analitica.csv')

## 8 · Dicionário e EDA mínima (pronto)
O dicionário vai para o README do grupo. A EDA responde: como y se distribui, e cada X parece se relacionar com y?

| Variável | Descrição | Papel |
|---|---|---|
| `CD_MUNICIPIO`, `NM_MUNICIPIO` | Código TSE e nome | ID |
| `ABST_2022_PCT`, `CRESC_ELEITORADO_PCT`, `DIST_CUIABA_KM` | Abstenção em 2022; variação do eleitorado de 2022 a julho de 2026; distância até Cuiabá | X |
| `PCT_16_24`, `PCT_65_MAIS`, `PCT_SUPERIOR_COMPLETO`, `LOG_ELEITORADO` | Perfil do eleitorado em julho de 2026 | X |
| `ELEITORADO_PERFIL`, `LATITUDE`, `LONGITUDE` | Contexto | já representados em outras colunas |
| `QT_APTOS`, `QT_ABSTENCOES` | Contagens de 2026 | **proibidas em X** (formam y) |
| `ABSTENCAO_PCT` | Abstenção no 1º turno de 2026 | **y** |

In [ ]:
import matplotlib.pyplot as plt

X_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
          'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
print('nulos por coluna:', int(base.isna().sum().sum()))
fig, eixos = plt.subplots(1, 4, figsize=(18, 3.6))
eixos[0].hist(base['ABSTENCAO_PCT'], bins=20, color='#1f6feb', alpha=0.8); eixos[0].set_title('Distribuição de y')
for ax, col in zip(eixos[1:], ['ABST_2022_PCT', 'DIST_CUIABA_KM', 'CRESC_ELEITORADO_PCT']):
    ax.scatter(base[col], base['ABSTENCAO_PCT'], s=14, alpha=0.6)
    ax.set_title(f"{col} (r = {base[col].corr(base['ABSTENCAO_PCT']):.2f})"); ax.set_xlabel(col)
plt.tight_layout(); plt.show()
base[X_COLS + ['ABSTENCAO_PCT']].corr()['ABSTENCAO_PCT'].round(2)

**Pergunta para discussão:** a abstenção de 2022 tem correlação de 0,77 com a de 2026. Por que ela pode entrar em X, se é "abstenção" como y?

**Resposta esperada:** porque é de **outra eleição**, conhecida quatro anos antes. A regra é: só entra em X o que se sabe no momento em que a previsão seria feita. A de 2022 se sabe; as contagens de 2026, não.

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('perfil: 142 municípios com os percentuais', lambda: len(m) == 142 and 'PCT_16_24' in base and base['PCT_16_24'].between(5, 25).all()),
    conferir('resultados: 142 municípios em 2026 e 141 em 2022', lambda: len(atual) == 142 and len(anterior) == 141),
    conferir('abstenção média em 2026: 23,45%', lambda: round(base['ABSTENCAO_PCT'].mean(), 2) == 23.45),
    conferir('abstenção média em 2022: 25,94%', lambda: round(base['ABST_2022_PCT'].mean(), 2) == 25.94),
    conferir('maior distância até Cuiabá: 831 km', lambda: round(base['DIST_CUIABA_KM'].max()) == 831),
    conferir('Boa Esperança do Norte retirada; 141 municípios', lambda: len(base) == 141 and 'BOA ESPERANÇA DO NORTE' not in set(base['NM_MUNICIPIO'])),
    conferir('base_analitica.csv salva com 15 colunas', lambda: pd.read_csv(SAIDA / 'base_analitica.csv').shape == (141, 15)),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist do check-in de 10/11
- [ ] O ETL está em **código** no repositório e gera `base_analitica.csv` a partir das fontes brutas.
- [ ] Cada fonte tem link e data de download no README.
- [ ] Dicionário com variável, tipo, fonte e papel (X, y, ID ou descarte).
- [ ] EDA mínima: distribuição de y, nulos e 3 gráficos de relação com y.
- [ ] As decisões de limpeza (como retirar um município) estão escritas.